In [3]:
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnableParallel, branch
from dataclasses import Field

from  langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from openai import OpenAI
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel
from typing import Literal, TypedDict
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END
from IPython.display import Image,display, Markdown



# This function will load all the variables from the .env file and will
# make them available in the os.environ dictionary (env variables)
load_dotenv()
ssl_cert_file=""

# Set SSL certificate for requests library (used by tiktoken)
os.environ['REQUESTS_CA_BUNDLE'] = ssl_cert_file
os.environ['SSL_CERT_FILE'] = ssl_cert_file

client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()

from langchain_community.document_loaders import PyPDFLoader


# Load new document
loader = PyPDFLoader("../../Docs/fabric-admin.pdf")
new_docs = loader.load()
new_docs

# DSX Embedding Interface (as per documentation) with batch processing
from langchain_core.embeddings import Embeddings
from typing import List

class DSXEmbeddingInterface(Embeddings):
      
    def __init__(self, url, model_name, http_client, default_headers, batch_size=32):
        self.client = OpenAI(
            base_url=url,
            http_client=http_client,
            api_key='not-empty',  # This is replaced with the token generation based on the authentication provider, but cannot be left blank due to a check in the OpenAI client
            default_headers=default_headers
        )
        self.model_name = model_name
        self.batch_size = batch_size
  
      
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """Embed search docs with batch processing."""
        all_embeddings = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i:i + self.batch_size]
            resp = self.client.embeddings.create(model=self.model_name, input=batch)
            all_embeddings.extend([a.embedding for a in resp.data])
        return all_embeddings
  
  
    def embed_query(self, text: str) -> List[float]:
        """Embed query text."""
        return self.embed_documents([text])[0]

# Create embedding model with DSX interface
embedding_model = DSXEmbeddingInterface(
    url="",
    model_name="",
    http_client=httpx.Client(verify=ssl.create_default_context(cafile=ssl_cert_file)),
    default_headers={
        "x-correlation-id": str(uuid.uuid4()),
        'accept': '*/*',
        'Content-Type': 'application/json',
        'Authorization': f"Basic {base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode()}"
    },
    batch_size=32  # Maximum allowed batch size
)

# Simple numpy-based vector store with persistence and add functionality
import numpy as np
import json

class SimpleVectorStore:
    def __init__(self, documents=None, embedding_model=None, persist_directory=None):
        self.documents = documents if documents is not None else []
        self.embedding_model = embedding_model
        self.embeddings = None
        self.persist_directory = persist_directory
        
    def add_documents(self):
        """Embed and add documents to the store."""
        texts = [doc.page_content for doc in self.documents]
        new_embeddings = np.array(self.embedding_model.embed_documents(texts))
        
        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            # Append new embeddings to existing ones
            self.embeddings = np.vstack([self.embeddings, new_embeddings])
        
    def add_documents_to_existing(self, new_documents):
        """Add new documents to an existing vector store."""
        texts = [doc.page_content for doc in new_documents]
        new_embeddings = np.array(self.embedding_model.embed_documents(texts))
        
        # Append new documents
        self.documents.extend(new_documents)
        
        # Append new embeddings
        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])
        
    def similarity_search(self, query, k=3):
        query_embedding = np.array(self.embedding_model.embed_query(query))
        # Manual cosine similarity calculation
        query_norm = np.linalg.norm(query_embedding)
        doc_norms = np.linalg.norm(self.embeddings, axis=1)
        similarities = np.dot(self.embeddings, query_embedding) / (doc_norms * query_norm)
        top_k_indices = np.argsort(similarities)[::-1][:k]
        return [self.documents[i] for i in top_k_indices]
    
    def save(self, directory=None):
        """Save the vector store to disk."""
        save_dir = directory or self.persist_directory
        if not save_dir:
            raise ValueError("No persist directory specified")
        
        os.makedirs(save_dir, exist_ok=True)
        
        # Save embeddings
        np.save(os.path.join(save_dir, "embeddings.npy"), self.embeddings)
        
        # Save documents as JSON
        docs_data = [{"page_content": doc.page_content, "metadata": doc.metadata} for doc in self.documents]
        with open(os.path.join(save_dir, "documents.json"), "w") as f:
            json.dump(docs_data, f)
        
        print(f"Vector store saved to {save_dir}")
    
    @classmethod
    def load(cls, directory, embedding_model):
        """Load the vector store from disk."""
        # Load embeddings
        embeddings = np.load(os.path.join(directory, "embeddings.npy"))
        
        # Load documents
        with open(os.path.join(directory, "documents.json"), "r") as f:
            docs_data = json.load(f)
        
        from langchain_core.documents import Document
        documents = [Document(page_content=doc["page_content"], metadata=doc["metadata"]) for doc in docs_data]
        
        # Create vector store instance
        vector_store = cls(documents=documents, embedding_model=embedding_model, persist_directory=directory)
        vector_store.embeddings = embeddings
        
        print(f"Vector store loaded from {directory}")
        return vector_store

# Define persist directory (absolute path to project root)
persist_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "vector_store"))

# Check if persisted vector store exists
if os.path.exists(persist_dir) and os.path.exists(os.path.join(persist_dir, "embeddings.npy")):
    # Load existing vector store
    vectorstore = SimpleVectorStore.load(persist_dir, embedding_model)
    print(f"Loaded existing vector store with {len(vectorstore.documents)} documents")
    
    # Add new documents
    vectorstore.add_documents_to_existing(new_docs)
    print(f"Added {len(new_docs)} new documents. Total documents: {len(vectorstore.documents)}")
else:
    # Create new vector store with the new document
    vectorstore = SimpleVectorStore(new_docs, embedding_model, persist_directory=persist_dir)
    vectorstore.add_documents()
    print(f"Created new vector store with {len(new_docs)} documents")

# Save vector store
vectorstore.save()
print("Vector store saved to disk")

# Test similarity search
query = "What is Fabric OneLake?"
results = vectorstore.similarity_search(query, k=3)
response = llm_openai.invoke(f"{query} You can answer using the following context: {results}")
print(response.content)

Vector store loaded from C:\projects\personal\langchn_practise\vector_store
Loaded existing vector store with 290 documents
Added 290 new documents. Total documents: 580
Vector store saved to C:\projects\personal\langchn_practise\vector_store
Vector store saved to disk
**Fabric OneLake** is the unified, cloud‑native data lake that underpins Microsoft Fabric. It is the single storage repository where all Fabric data (lake, warehouse, lakehouse, and analytics assets) lives, and it can be accessed both from inside Fabric (e.g., Spark, Data Engineering, Data Warehouse) and from external applications (e.g., custom apps using Azure Data Lake Storage APIs, Databricks, or the OneLake File Explorer).  

Key points from the Microsoft Fabric documentation:

- **Tenant‑level configuration** – OneLake settings are managed in the **tenant settings** section of the Fabric admin portal. 【Document page 290】  
- **External access** – Users can read and write OneLake data with apps outside of Fabric, suc